In [ ]:
# データセットの統計情報を表示するプログラム

In [ ]:
# Google Driveをマウント
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import re
import os

# ---------------------------------------------------------
# 論文「表1」作成用：データセット統計量算出スクリプト
# ---------------------------------------------------------

def load_data(file_path):
    """
    ファイルの拡張子を判別してデータを読み込む関数
    """
    if not os.path.exists(file_path):
        print(f"Warning: File not found '{file_path}'. Skipping.")
        return None

    if file_path.endswith('.csv'):
        return pd.read_csv(file_path)
    elif file_path.endswith('.xlsx') or file_path.endswith('.xls'):
        return pd.read_excel(file_path)
    else:
        raise ValueError(f"Unsupported file format: {file_path}")

def calculate_stats(df, label):
    """
    統計情報を計算するコア関数
    """
    stats = {}
    stats['Dataset / Split'] = label
    stats['Count (N)'] = len(df)

    # --- カラム名の自動検出ロジック (ここを修正しました) ---
    # 日本語列の候補 (combined_ja, expert_ja を優先的に探します)
    ja_candidates = ['combined_ja', 'expert_ja', 'japanese', 'ja', 'jp', 'source']
    # 英語列の候補
    en_candidates = ['english', 'en', 'target']

    ja_col = next((c for c in df.columns if c.lower() in ja_candidates), None)
    en_col = next((c for c in df.columns if c.lower() in en_candidates), None)

    if not ja_col or not en_col:
        print(f"Error: Columns not found in {label}. Found: {df.columns}")
        return None

    # データ型変換
    ja_texts = df[ja_col].astype(str).fillna("")
    en_texts = df[en_col].astype(str).fillna("")

    # 1. 平均文長 (Average Length)
    stats['Avg Length (JA chars)'] = ja_texts.apply(len).mean()
    stats['Avg Length (EN words)'] = en_texts.apply(lambda x: len(x.split())).mean()
    stats['Avg Length (EN chars)'] = en_texts.apply(len).mean()

    # 2. 語彙数 (Vocabulary Size - EN)
    en_words = set()
    for text in en_texts:
        # 記号を除去し、小文字化してから単語セットに追加
        clean_text = re.sub(r'[^\w\s]', '', text.lower())
        en_words.update(clean_text.split())
    stats['Vocab Size (EN)'] = len(en_words)

    return stats

# ---------------------------------------------------------
# メイン処理（ご自身のパスに合わせてください．）
# ---------------------------------------------------------

# ベースとなるディレクトリパス
base_dir = 'sss'

# ファイルパスの設定 (これまでの工程で作成したファイル名に合わせました)
file_paths = {
    # 元の大規模データ (必要であればコメントアウトを外してください)
    # "All Data (Source Pool)":       base_dir + "data_with_all_scores_completed.csv",

    "Model 1 Train (High Quality)": base_dir + "train_data_model1_custom_3k.csv",     # 高品質3k
    "Model 2 Train (Random)":       base_dir + "train_data_model2_random_3k.csv",     # ランダム3k
    "Model 3 Train (Combined)":     base_dir + "train_data_model3_combined_6k.csv",     # 混合6k
    "Validation Set":               base_dir + "validation_set_500.csv",  # 検証用500
    "Test Set":                     base_dir + "test_set_500.csv"         # テスト用500
}

# --- 計算実行 ---
results = []
print("--- Calculating Statistics ---")

for label, path in file_paths.items():
    try:
        df = load_data(path)
        if df is not None:
            stats = calculate_stats(df, label)
            if stats:
                results.append(stats)
    except Exception as e:
        print(f"Error processing {label}: {e}")

# --- 結果の表示 ---
if results:
    results_df = pd.DataFrame(results)
    pd.options.display.float_format = '{:.1f}'.format

    print("\n" + "="*80)
    print("【論文 表2 (Table 2) 用データ統計 EN-JA】")
    print("="*80)
    # 表示する列の順序
    cols = ['Dataset / Split', 'Count (N)', 'Avg Length (JA chars)', 'Avg Length (EN words)', 'Avg Length (EN chars)', 'Vocab Size (EN)']
    print(results_df[cols].to_string(index=False))
    print("="*80)
else:
    print("データが正しく読み込めませんでした。パスを確認してください。")